# AI Math

## 사전 준비

### 패키지 설치

In [ ]:
!pip install pandas seaborn numpy matplotlib tqdm

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm

### 데이터셋 가져오기

In [ ]:
df = sns.load_dataset('diamonds')
df.info()

In [ ]:
categorical_cols = df.select_dtypes(include=['category']).columns.tolist()
categorical_cols

In [ ]:
continuous_cols = df.select_dtypes(include=['number']).columns.tolist()
continuous_cols

In [ ]:
df.describe()

In [ ]:
### 데이터셋 표준화

In [ ]:
xmat_raw = df[continuous_cols].values

# 평균 계산
mu = xmat_raw.mean(axis=0)
# 표준편차 계산
std = xmat_raw.std(axis=0)

# 표준편차가 0인 데이터의 표준편차를 작은 값(epsillon)으로 교체
epsillon = 1e-8
import numpy as np
std = np.where(std == 0, epsillon, std)

# 표준화
xmat_norm = (xmat_raw - mu) / std
xmat_norm

## 수학으로 정답 구하기

### 비용함수

결정한 파라미터($\theta$)를 바탕으로 실제 데이터($X$)를 가지고 추측한 결과($y$)와 실제 답($\hat{y}$)을 바탕으로
파라미터가 얼마나 잘 만들어졌는지 판단하는 기준

### 평균 제곱 오차 (Mean Square Error)

추측한 결과와 실제 답의 오차의 제곱의 평균을 비용함수로 사용하는 방법

$J(\theta) = \frac{1}{n}\sum_{i=1}^{n}(\hat{y_i} - \theta x)^2$

### 정규방정식을 이용한 풀이

비용함수 MSE는 최종적으로 $\theta$에 대한 2차 방정식으로 미분과 선형대수학으로 
비용함수가 최소가 되는 $\theta$를 구할 수 있다.

$\mathbf{\theta} = (\mathbf{X}^\mathsf{T}\mathbf{X})^{-1}\mathbf{X}^\mathsf{T}\mathbf{y}$

In [ ]:
import pandas as pd
xmat_df = pd.DataFrame(xmat_norm, columns=continuous_cols)
# 가격을 예측하는게 목적임으로 가격은 분리
xmat = xmat_df.drop(labels='price', axis=1).values
y = xmat_df['price'].values

# 절편항 추가 (Feature가 없을 때 추측값)
m, _ = xmat_norm.shape
xmat_wbeta = np.c_[np.ones((m, 1)), xmat_norm]

# 정규방정식으로 theta 구하기
xmat_trans = xmat_wbeta.T
theta = np.linalg.inv(xmat_trans @ xmat_wbeta) @ xmat_trans @ y
theta

In [ ]:
# 구한 theta의 평균제곱오차
y_result = xmat_wbeta @ theta
mse = np.mean((y_result - y) ** 2)
print(f'mse for normal equation: {mse}')

In [ ]:
# 시각화 함수
def plot_prediction(y_true, y_pred):
    import matplotlib.pyplot as plt
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    assert y_true.shape == y_pred.shape, f"Size mismatch between y_true and y_pred"

    fig, ax = plt.subplots(figsize=(6, 4))

    # 회귀선
    sns.scatterplot(x=y_true, y=y_pred,
                    alpha=0.5, label="Model Prediction", ax=ax)

    # 이상적인 예측선
    sns.lineplot(x=[y_pred.min(), y_pred.max()],
                 y=[y_true.min(), y_true.max()],
                 label="Ideal Regression", linestyle="--", color="red")

    ax.set_xlabel('Actual Price')
    ax.set_ylabel('Predicted Price')
    ax.set_title('Actual vs Predicted Price')
    fig.tight_layout()
    plt.show()

In [ ]:
plot_prediction(y, y_result)

### 최소제곱법을 이용한 풀이

역행렬이 존재하지 않을 때 의사역행렬을 활용해 정답에 가까운 $\theta$를 구하는 방법

In [ ]:
# np.linalg.lstsq로 최소제곱법으로 theta를 구할 수 있음
theta_lstsq, _, _, _ = np.linalg.lstsq(xmat_wbeta, y, rcond=None)
y_result = xmat_wbeta @ theta_lstsq
mse = np.mean((y_result - y) ** 2)
print(f'mse for lstsq: {mse}')
plot_prediction(y, y_result)

## 경사하강법

데이터의 양이 늘어나게 될 경우 수학적 풀이는 필요한 연산 횟수가 비현실적으로 많아진다.

경사하강법은 실험과 평가를 통해 최적의 $\theta$를 찾아내는 방법

1. 적당한 첫 $\theta$를 정한다.
2. $\theta$ 지점에서의 비용함수의 경사(미분값)을 구한다.
3. 경사도를 바탕으로 $\theta$를 조정한다.
4. 이를 정해진 횟수만큼, 또는 다른 기준을 만족할 때 까지 반복한다.

In [ ]:
# 초기 theta
_, n = xmat_norm.shape 
theta = np.zeros(n+1)
# 학습률
alpha = 0.01
# 반복 횟수
iterations = 5000
# 비용 계산 결과 기록
loss_history = []

# 정해진 iterations 수만큼
for i in tqdm(range(iterations)):
    # 예측값 계산
    y_pred = (xmat_wbeta @ theta).flatten()

    # 오차를 바탕으로 MSE 계산 및 기록
    error = y_pred - y
    mse = np.mean(error**2)
    loss_history.append(mse)

    # 식을 토대로 경사(gradient) 계산
    gradient = (2/m) * xmat_wbeta.T @ error

    # 학습률을 가지고 theta 업데이트
    theta -= alpha * gradient

print(loss_history[100:110])
print("최종 θ:", theta.flatten())
print("최종 MSE:", loss_history[-1])

import matplotlib.pyplot as plt
plt.plot(loss_history)
plt.xlabel("Iteration", size="large")
plt.ylabel("Mean Squared Error", size="large")
plt.title("Loss Curve", size="large")
plt.show()